# Publish-Subscribe e Arquitetura Baseada em Eventos: exemplo executável

**Guia Aberto de Arquitetura de Software** · [Página do tópico](https://ifpe-belojardim-es.github.io/guia-arquitetura-de-software/content/07-publish-subscribe-e-eventos/)

Autores: Antonio Mário Jesus Vidal Leite, David Soares da Silva, Ian Képler dos Santos Araújo, João Gabriel Marcos da Silva, João Lucas da Silva Soares Oliveira, Lucas Matheus Elias Silva Teixeira dos Santos · Licença: CC BY-NC 4.0

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IFPE-BeloJardim-ES/guia-arquitetura-de-software/blob/main/content/07-publish-subscribe-e-eventos/notebook.ipynb)

---

Este notebook implementa a loja virtual do texto **duas vezes**: uma com chamadas diretas e outra com publish-subscribe. As duas versões passam pelos mesmos cenários e são medidas com os mesmos critérios.

Na segunda metade, quebramos de propósito a terceira restrição do estilo (*o assinante não assume entrega única nem ordem*) e medimos o estrago. Depois aplicamos as duas correções discutidas no texto: receptor idempotente e versão do evento.

## 0. Dependências

Nada para instalar. Tudo usa só a biblioteca padrão do Python, então roda igual no Colab e em qualquer máquina com Python 3.9 ou mais novo.

Os serviços externos (e-mail, fidelidade, painel) são simulados com `time.sleep`, para que o tempo de resposta possa ser medido. O broker é escrito do zero e roda num único processo: ele só entrega as mensagens quando chamamos `entregar_pendentes()`. Isso deixa os resultados reproduzíveis, coisa que um broker real, com threads e rede, não garante.

In [1]:
import sys
import time
import inspect
import itertools
from collections import deque
from dataclasses import dataclass, field
from typing import Callable, Dict, List

print(f"Python {sys.version.split()[0]}")

Python 3.13.18


## 1. O problema

Quando o cliente clica em "finalizar compra", a loja precisa:

1. baixar o estoque;
2. mandar um e-mail de confirmação;
3. creditar pontos de fidelidade (1 ponto por real);
4. atualizar a tela "Meus pedidos".

Abaixo estão os quatro serviços, com tempos de resposta simulados. O e-mail é de longe o mais lento, como costuma ser na prática. Cada serviço tem uma "chave" para simular falha: `fora_do_ar` no e-mail e `em_manutencao` na fidelidade.

In [2]:
class ServicoIndisponivel(Exception):
    "Falha de um serviço externo: servidor fora do ar, banco sem resposta etc."


class Estoque:
    def __init__(self, inicial):
        self.quantidades = dict(inicial)

    def baixar(self, produto, quantidade):
        time.sleep(0.01)
        if self.quantidades.get(produto, 0) < quantidade:
            raise ValueError(f"sem estoque de {produto}")
        self.quantidades[produto] -= quantidade

    def devolver(self, produto, quantidade):
        self.quantidades[produto] += quantidade


class Notificacao:
    def __init__(self):
        self.fora_do_ar = False
        self.enviados = []

    def enviar(self, cliente, mensagem):
        time.sleep(0.30)                      # servidor de e-mail lento
        if self.fora_do_ar:
            raise ServicoIndisponivel("servidor de e-mail fora do ar")
        self.enviados.append((cliente, mensagem))


class Fidelidade:
    def __init__(self):
        self.em_manutencao = False
        self.pontos = {}

    def creditar(self, cliente, valor):
        time.sleep(0.07)
        if self.em_manutencao:
            raise ServicoIndisponivel("fidelidade em manutenção")
        self.pontos[cliente] = self.pontos.get(cliente, 0) + int(valor)


class Painel:
    "A tela 'Meus pedidos'."

    def __init__(self):
        self.status = {}

    def registrar(self, pedido_id, status):
        time.sleep(0.05)
        self.status[pedido_id] = status


class NotaFiscal:
    "Pedido do setor fiscal que chega depois. Só será usado na seção 2.3."

    def __init__(self):
        self.emitidas = []

    def emitir(self, pedido_id, valor):
        time.sleep(0.02)
        self.emitidas.append((pedido_id, valor))


_numero_pedido = itertools.count(1)

def novo_pedido(cliente="Ana", produto="caneca", quantidade=1, valor=50.0):
    return {"id": next(_numero_pedido), "cliente": cliente, "produto": produto,
            "quantidade": quantidade, "valor": valor, "versao": 1}


def segundos(t):
    "Formata um tempo em segundos com vírgula decimal."
    return f"{t:.2f} s".replace(".", ",")

### 1.1 A versão com chamadas diretas

O jeito mais direto: a função de venda chama os quatro serviços, um depois do outro. É o mesmo código mostrado no texto.

In [3]:
class LojaAcoplada:
    def __init__(self, estoque, notificacao, fidelidade, painel):
        self.estoque = estoque
        self.notificacao = notificacao
        self.fidelidade = fidelidade
        self.painel = painel

    def finalizar_pedido(self, pedido):
        self.estoque.baixar(pedido["produto"], pedido["quantidade"])
        self.notificacao.enviar(pedido["cliente"], f"Pedido {pedido['id']} confirmado")
        self.fidelidade.creditar(pedido["cliente"], pedido["valor"])
        self.painel.registrar(pedido["id"], "confirmado")
        return "confirmado"


def montar_loja_acoplada():
    s = {"estoque": Estoque({"caneca": 10}), "notificacao": Notificacao(),
         "fidelidade": Fidelidade(), "painel": Painel()}
    return LojaAcoplada(**s), s


loja, s = montar_loja_acoplada()
pedido = novo_pedido()
inicio = time.perf_counter()
resposta = loja.finalizar_pedido(pedido)
duracao = time.perf_counter() - inicio

print(f"Resposta ao cliente: {resposta} (depois de {segundos(duracao)})")
print(f"Canecas em estoque:  {s['estoque'].quantidades['caneca']}")
print(f"E-mails enviados:    {s['notificacao'].enviados}")
print(f"Pontos:              {s['fidelidade'].pontos}")
print(f"Painel:              {s['painel'].status}")

Resposta ao cliente: confirmado (depois de 0,43 s)
Canecas em estoque:  9
E-mails enviados:    [('Ana', 'Pedido 1 confirmado')]
Pontos:              {'Ana': 50}
Painel:              {1: 'confirmado'}


Funciona. Mas repare em três coisas, que a seção 2 vai medir:

- a loja **conhece** os quatro serviços: eles estão no construtor;
- o cliente esperou a **soma** dos quatro tempos, inclusive o do e-mail, que não muda nada na venda;
- se qualquer um dos três últimos serviços falhar, a exceção sobe até a venda.

## 2. Implementação seguindo o estilo

### 2.1 Evento e broker

Um **evento** descreve um fato que já aconteceu, por isso o nome no passado: `PedidoCriado`. Ele carrega os dados de que os assinantes precisam (*event-carried state transfer* [4]) e um **identificador único**, que vai ser essencial na seção 3.

O **broker** guarda uma fila por assinante. Publicar um evento põe uma cópia em cada fila interessada; nada é entregue na hora. As regras de entrega são as de um broker "pelo menos uma vez" [3], [11]:

- se o assinante lança exceção, a mensagem **não foi confirmada** e volta para o **fim** da fila;
- depois de `max_tentativas` falhas, ela vai para a **dead letter**;
- se o assinante estiver desconectado, as mensagens **esperam** na fila dele.

Todo passo vai para `broker.log`. Num sistema com eventos, esse registro é a única forma de reconstruir o que aconteceu numa venda.

In [4]:
_seq_evento = itertools.count(1)


@dataclass
class Evento:
    tipo: str
    dados: dict
    id: str = field(default_factory=lambda: f"evt-{next(_seq_evento)}")


@dataclass
class Mensagem:
    evento: Evento
    tentativas: int = 0


class Broker:
    def __init__(self, max_tentativas=3):
        self.max_tentativas = max_tentativas
        self.assinaturas: Dict[str, List[str]] = {}   # tipo de evento -> filas
        self.filas: Dict[str, deque] = {}             # fila -> mensagens
        self.handlers: Dict[str, Callable] = {}       # fila -> quem trata
        self.desconectados = set()
        self.dead_letter: List[tuple] = []
        self.log: List[str] = []

    # --- lado do assinante -------------------------------------------------
    def assinar(self, tipo, fila, handler):
        self.assinaturas.setdefault(tipo, [])
        if fila not in self.assinaturas[tipo]:
            self.assinaturas[tipo].append(fila)
        self.filas.setdefault(fila, deque())
        self.handlers[fila] = handler

    def desconectar(self, fila):
        self.desconectados.add(fila)

    def conectar(self, fila):
        self.desconectados.discard(fila)

    # --- lado do publicador ------------------------------------------------
    def publicar(self, evento):
        # O publicador não recebe nada de volta: nem quem assinou, nem quantos são.
        for fila in self.assinaturas.get(evento.tipo, []):
            self.filas[fila].append(Mensagem(evento))
        self.log.append(f"publicado {evento.id} {evento.tipo}")

    # --- entrega -------------------------------------------------------------
    def entregar_pendentes(self):
        for nome, fila in self.filas.items():
            if nome in self.desconectados:
                continue                       # as mensagens esperam na fila
            while fila:
                msg = fila.popleft()
                msg.tentativas += 1
                ev = msg.evento
                try:
                    self.handlers[nome](ev)
                except Exception as erro:
                    self.log.append(f"falhou    {ev.id} {ev.tipo} -> {nome} "
                                    f"(tentativa {msg.tentativas}: {erro})")
                    if msg.tentativas >= self.max_tentativas:
                        self.dead_letter.append((nome, ev, str(erro)))
                        self.log.append(f"dead letter {ev.id} {ev.tipo} -> {nome}")
                    else:
                        fila.append(msg)       # nova tentativa vai para o FIM da fila
                else:
                    self.log.append(f"entregue  {ev.id} {ev.tipo} -> {nome} "
                                    f"(tentativa {msg.tentativas})")

    def pendentes(self, fila):
        return len(self.filas[fila])

    def mostrar_log(self):
        for linha in self.log:
            print("  " + linha)

### 2.2 A loja e os assinantes

A loja só conhece o estoque e o broker. O estoque **continua sendo chamada direta**: a loja não pode responder "confirmado" sem saber se a caneca existe. O critério do texto: se a resposta ao cliente depende do resultado, é chamada direta; se é reação ao que já aconteceu, é evento [2].

O pedido leva um campo `versao`, que a loja incrementa a cada mudança. Ele só vai importar na seção 3.2.

Os assinantes são pequenos adaptadores: recebem o evento e chamam o serviço. Nenhum deles chama a loja de volta, porque o evento já traz tudo o que precisam.

In [5]:
class LojaComEventos:
    def __init__(self, estoque, broker):
        self.estoque = estoque
        self.broker = broker

    def finalizar_pedido(self, pedido):
        self.estoque.baixar(pedido["produto"], pedido["quantidade"])
        self.broker.publicar(Evento("PedidoCriado", dict(pedido)))
        return "confirmado"

    def cancelar_pedido(self, pedido):
        self.estoque.devolver(pedido["produto"], pedido["quantidade"])
        pedido["versao"] += 1
        self.broker.publicar(Evento("PedidoCancelado", dict(pedido)))
        return "cancelado"


STATUS = {"PedidoCriado": "confirmado", "PedidoCancelado": "cancelado"}

def assinante_notificacao(notificacao):
    def tratar(evento):
        d = evento.dados
        notificacao.enviar(d["cliente"], f"Pedido {d['id']} confirmado")
    return tratar

def assinante_fidelidade(fidelidade):
    def tratar(evento):
        fidelidade.creditar(evento.dados["cliente"], evento.dados["valor"])
    return tratar

def assinante_painel(painel):
    def tratar(evento):
        painel.registrar(evento.dados["id"], STATUS[evento.tipo])
    return tratar

def assinante_nota_fiscal(nota):
    def tratar(evento):
        nota.emitir(evento.dados["id"], evento.dados["valor"])
    return tratar


def montar_loja_com_eventos():
    s = {"estoque": Estoque({"caneca": 10}), "notificacao": Notificacao(),
         "fidelidade": Fidelidade(), "painel": Painel()}
    broker = Broker()
    broker.assinar("PedidoCriado", "notificacao", assinante_notificacao(s["notificacao"]))
    broker.assinar("PedidoCriado", "fidelidade", assinante_fidelidade(s["fidelidade"]))
    painel = assinante_painel(s["painel"])
    broker.assinar("PedidoCriado", "painel", painel)
    broker.assinar("PedidoCancelado", "painel", painel)
    return LojaComEventos(s["estoque"], broker), broker, s


loja, broker, s = montar_loja_com_eventos()
pedido = novo_pedido()
inicio = time.perf_counter()
resposta = loja.finalizar_pedido(pedido)
duracao = time.perf_counter() - inicio
print(f"Resposta ao cliente: {resposta} (depois de {segundos(duracao)})")
print(f"Pontos antes da entrega: {s['fidelidade'].pontos}   <- consistência eventual")

broker.entregar_pendentes()
print(f"Pontos depois da entrega: {s['fidelidade'].pontos}")
print(f"Painel: {s['painel'].status}")
print("\nLog do broker:")
broker.mostrar_log()

Resposta ao cliente: confirmado (depois de 0,01 s)
Pontos antes da entrega: {}   <- consistência eventual
Pontos depois da entrega: {'Ana': 50}
Painel: {2: 'confirmado'}

Log do broker:
  publicado evt-1 PedidoCriado
  entregue  evt-1 PedidoCriado -> notificacao (tentativa 1)
  entregue  evt-1 PedidoCriado -> fidelidade (tentativa 1)
  entregue  evt-1 PedidoCriado -> painel (tentativa 1)


Repare na linha "Pontos antes da entrega". Entre a resposta ao cliente e o processamento dos assinantes existe uma janela em que o pedido está confirmado e os pontos ainda não. Isso é a **consistência eventual**: os dados chegam ao estado certo, mas não no mesmo instante.

A partir daqui, as duas versões passam pelos mesmos cenários. Os resultados vão sendo guardados em `resultados` para a tabela final.

In [6]:
resultados = {}

def servicos_conhecidos(classe):
    "Os parâmetros do construtor são as dependências que a loja precisa receber."
    return [p for p in inspect.signature(classe.__init__).parameters if p != "self"]

for classe in (LojaAcoplada, LojaComEventos):
    deps = servicos_conhecidos(classe)
    print(f"{classe.__name__:15} conhece {len(deps)}: {', '.join(deps)}")

resultados["Serviços que a loja precisa conhecer"] = (
    len(servicos_conhecidos(LojaAcoplada)), len(servicos_conhecidos(LojaComEventos)))

LojaAcoplada    conhece 4: estoque, notificacao, fidelidade, painel
LojaComEventos  conhece 2: estoque, broker


### 2.3 Chega um pedido novo: emitir nota fiscal

O setor fiscal quer uma nota a cada pedido. Na versão acoplada, não tem jeito: alguém abre a loja, acrescenta um parâmetro no construtor e mais uma linha em `finalizar_pedido`. A classe abaixo é essa loja editada.

In [7]:
class LojaAcopladaComNota:
    def __init__(self, estoque, notificacao, fidelidade, painel, nota_fiscal):  # mudou
        self.estoque = estoque
        self.notificacao = notificacao
        self.fidelidade = fidelidade
        self.painel = painel
        self.nota_fiscal = nota_fiscal                                         # nova

    def finalizar_pedido(self, pedido):
        self.estoque.baixar(pedido["produto"], pedido["quantidade"])
        self.notificacao.enviar(pedido["cliente"], f"Pedido {pedido['id']} confirmado")
        self.fidelidade.creditar(pedido["cliente"], pedido["valor"])
        self.painel.registrar(pedido["id"], "confirmado")
        self.nota_fiscal.emitir(pedido["id"], pedido["valor"])                 # nova
        return "confirmado"

print(f"LojaAcopladaComNota conhece {len(servicos_conhecidos(LojaAcopladaComNota))} serviços")

LojaAcopladaComNota conhece 5 serviços


Na versão com eventos, a nota fiscal entra como **mais um assinante** de `PedidoCriado`. A classe `LojaComEventos` é exatamente a mesma, sem uma linha alterada.

In [8]:
loja, broker, s = montar_loja_com_eventos()
nota = NotaFiscal()
broker.assinar("PedidoCriado", "nota_fiscal", assinante_nota_fiscal(nota))   # só isso

loja.finalizar_pedido(novo_pedido())
broker.entregar_pendentes()

print(f"Notas emitidas: {nota.emitidas}")
print(f"A loja ainda é a mesma classe? {type(loja) is LojaComEventos}")
print(f"LojaComEventos conhece {len(servicos_conhecidos(LojaComEventos))} serviços")

resultados["Depois de incluir a nota fiscal"] = (
    f"{len(servicos_conhecidos(LojaAcopladaComNota))} (a loja foi editada)",
    f"{len(servicos_conhecidos(LojaComEventos))} (a loja não mudou)")

Notas emitidas: [(3, 50.0)]
A loja ainda é a mesma classe? True
LojaComEventos conhece 2 serviços


### 2.4 Tempo de resposta ao cliente

Medimos só o que o cliente espera: o tempo de `finalizar_pedido`. Tiramos a média de três vendas em cada versão. Os números exatos variam de máquina para máquina; a diferença de ordem de grandeza, não.

In [9]:
def tempo_medio(loja, vendas=3):
    tempos = []
    for _ in range(vendas):
        inicio = time.perf_counter()
        loja.finalizar_pedido(novo_pedido())
        tempos.append(time.perf_counter() - inicio)
    return sum(tempos) / len(tempos)

t_acoplada = tempo_medio(montar_loja_acoplada()[0])
t_eventos = tempo_medio(montar_loja_com_eventos()[0])

print(f"Chamadas diretas: {segundos(t_acoplada)}  (estoque + e-mail + fidelidade + painel)")
print(f"Eventos:          {segundos(t_eventos)}  (só o estoque)")
print(f"A versão com eventos respondeu {t_acoplada / t_eventos:.0f} vezes mais rápido.")

resultados["Tempo de resposta ao cliente"] = (
    f"cerca de {segundos(t_acoplada)}", f"cerca de {segundos(t_eventos)}")

Chamadas diretas: 0,43 s  (estoque + e-mail + fidelidade + painel)
Eventos:          0,01 s  (só o estoque)
A versão com eventos respondeu 42 vezes mais rápido.


O trabalho dos assinantes não sumiu: ele acontece depois, em `entregar_pendentes()`. O ganho é que o cliente não precisa esperar por ele.

### 2.5 O servidor de e-mail cai

Na versão acoplada, a exceção do e-mail sobe até a venda. E repare na ordem das linhas em `finalizar_pedido`: o estoque já foi baixado quando o e-mail falha.

In [10]:
loja, s = montar_loja_acoplada()
s["notificacao"].fora_do_ar = True
try:
    loja.finalizar_pedido(novo_pedido())
    situacao_acoplada = "concluída"
except ServicoIndisponivel as erro:
    situacao_acoplada = f"recusada ({erro})"

print("CHAMADAS DIRETAS")
print(f"  Venda: {situacao_acoplada}")
print(f"  Canecas em estoque: {s['estoque'].quantidades['caneca']} de 10  <- a caneca já saiu")
print(f"  Pontos: {s['fidelidade'].pontos}   Painel: {s['painel'].status}")

CHAMADAS DIRETAS
  Venda: recusada (servidor de e-mail fora do ar)
  Canecas em estoque: 9 de 10  <- a caneca já saiu
  Pontos: {}   Painel: {}


In [11]:
loja, broker, s = montar_loja_com_eventos()
s["notificacao"].fora_do_ar = True
resposta = loja.finalizar_pedido(novo_pedido())
broker.entregar_pendentes()

tentativas_email = sum(1 for l in broker.log if l.startswith("falhou") and "notificacao" in l)
print("EVENTOS")
print(f"  Venda: {resposta}")
print(f"  Canecas em estoque: {s['estoque'].quantidades['caneca']} de 10")
print(f"  Pontos: {s['fidelidade'].pontos}   Painel: {s['painel'].status}")
print(f"  Tentativas de e-mail: {tentativas_email}   Dead letter: {len(broker.dead_letter)} mensagem(ns)")
print("\nLog do broker:")
broker.mostrar_log()

resultados["Venda com o e-mail fora do ar"] = (
    "recusada, mas a caneca já tinha saído do estoque",
    f"concluída; e-mail tentado {tentativas_email} vezes e enviado à dead letter")

EVENTOS
  Venda: confirmado
  Canecas em estoque: 9 de 10
  Pontos: {'Ana': 50}   Painel: {11: 'confirmado'}
  Tentativas de e-mail: 3   Dead letter: 1 mensagem(ns)

Log do broker:
  publicado evt-6 PedidoCriado
  falhou    evt-6 PedidoCriado -> notificacao (tentativa 1: servidor de e-mail fora do ar)
  falhou    evt-6 PedidoCriado -> notificacao (tentativa 2: servidor de e-mail fora do ar)
  falhou    evt-6 PedidoCriado -> notificacao (tentativa 3: servidor de e-mail fora do ar)
  dead letter evt-6 PedidoCriado -> notificacao
  entregue  evt-6 PedidoCriado -> fidelidade (tentativa 1)
  entregue  evt-6 PedidoCriado -> painel (tentativa 1)


A falha ficou isolada no assinante de notificação. Mas veja o outro lado: a venda seguiu e ninguém foi avisado. Se ninguém olhar a dead letter, esse e-mail simplesmente nunca chega. Na versão acoplada a falha era ruim, mas pelo menos era **visível**.

### 2.6 A fidelidade entra em manutenção durante três vendas

Agora o desacoplamento **no tempo** [1]: o serviço de fidelidade fica fora por um tempo e, nesse meio-tempo, três clientes compram.

In [12]:
loja, s = montar_loja_acoplada()
s["fidelidade"].em_manutencao = True
recusadas = 0
for _ in range(3):
    try:
        loja.finalizar_pedido(novo_pedido())
    except ServicoIndisponivel:
        recusadas += 1

canecas_fora = 10 - s["estoque"].quantidades["caneca"]
emails = len(s["notificacao"].enviados)
print("CHAMADAS DIRETAS")
print(f"  Vendas recusadas: {recusadas} de 3")
print(f"  Canecas que saíram do estoque: {canecas_fora}")
print(f"  E-mails de 'confirmado' enviados: {emails}   <- para vendas que foram recusadas")

CHAMADAS DIRETAS
  Vendas recusadas: 3 de 3
  Canecas que saíram do estoque: 3
  E-mails de 'confirmado' enviados: 3   <- para vendas que foram recusadas


In [13]:
loja, broker, s = montar_loja_com_eventos()
broker.desconectar("fidelidade")            # serviço em manutenção
for _ in range(3):
    loja.finalizar_pedido(novo_pedido())
broker.entregar_pendentes()

print("EVENTOS")
print(f"  Vendas concluídas: 3 de 3")
print(f"  Durante a manutenção: {broker.pendentes('fidelidade')} eventos esperando, "
      f"pontos = {s['fidelidade'].pontos}")

broker.conectar("fidelidade")               # serviço voltou
broker.entregar_pendentes()
pontos = s["fidelidade"].pontos.get("Ana", 0)
print(f"  Quando voltou: {broker.pendentes('fidelidade')} eventos esperando, pontos = {pontos}")

resultados["Fidelidade em manutenção durante 3 vendas"] = (
    f"{recusadas} vendas recusadas, {canecas_fora} canecas fora do estoque "
    f"e {emails} e-mails de \"confirmado\" enviados",
    f"3 vendas concluídas; {pontos} pontos creditados quando o serviço voltou")

EVENTOS
  Vendas concluídas: 3 de 3
  Durante a manutenção: 3 eventos esperando, pontos = {}
  Quando voltou: 0 eventos esperando, pontos = 150


### 2.7 Resumo das medições

Esta é a tabela que aparece no texto do tópico, gerada a partir das execuções acima.

In [14]:
print(f"{'Medição':42} | {'Chamadas diretas':52} | Eventos")
print("-" * 150)
for medicao, (acoplada, eventos) in resultados.items():
    print(f"{medicao:42} | {str(acoplada):52} | {eventos}")

Medição                                    | Chamadas diretas                                     | Eventos
------------------------------------------------------------------------------------------------------------------------------------------------------
Serviços que a loja precisa conhecer       | 4                                                    | 2
Depois de incluir a nota fiscal            | 5 (a loja foi editada)                               | 2 (a loja não mudou)
Tempo de resposta ao cliente               | cerca de 0,43 s                                      | cerca de 0,01 s
Venda com o e-mail fora do ar              | recusada, mas a caneca já tinha saído do estoque     | concluída; e-mail tentado 3 vezes e enviado à dead letter
Fidelidade em manutenção durante 3 vendas  | 3 vendas recusadas, 3 canecas fora do estoque e 3 e-mails de "confirmado" enviados | 3 vendas concluídas; 150 pontos creditados quando o serviço voltou


## 3. A violação: assinantes que ignoram a terceira restrição

Até aqui os assinantes funcionaram porque tivemos sorte: cada evento chegou uma vez e na ordem certa. Mas o broker é "pelo menos uma vez" e devolve as falhas para o fim da fila. Um assinante que assume entrega única e ordem está violando a terceira restrição do estilo. Vamos provocar as duas situações.

### 3.1 Entrega duplicada

O serviço de fidelidade credita os pontos, mas a **confirmação** de volta ao broker se perde na rede. Para o broker, a entrega falhou, então ele entrega de novo. O assinante não fez nada errado no trabalho em si; o erro foi não esperar uma repetição.

In [15]:
def perde_confirmacao_na_primeira(handler):
    "O assinante faz o trabalho, mas a confirmação ao broker se perde uma vez por evento."
    ja_perdeu = set()
    def tratar(evento):
        handler(evento)
        if evento.id not in ja_perdeu:
            ja_perdeu.add(evento.id)
            raise TimeoutError("confirmação perdida na rede")
    return tratar


def cenario_duplicata(montar_assinante):
    broker, fidelidade = Broker(), Fidelidade()
    loja = LojaComEventos(Estoque({"caneca": 10}), broker)
    broker.assinar("PedidoCriado", "fidelidade",
                   perde_confirmacao_na_primeira(montar_assinante(fidelidade)))
    loja.finalizar_pedido(novo_pedido(valor=50.0))
    broker.entregar_pendentes()
    entregas = sum(1 for l in broker.log if "-> fidelidade" in l)
    return fidelidade.pontos.get("Ana", 0), entregas, broker


pontos_ingenuo, entregas, broker = cenario_duplicata(assinante_fidelidade)
print(f"Pedido de R$ 50,00 -> {pontos_ingenuo} pontos  (o broker entregou {entregas} vezes)")
broker.mostrar_log()

Pedido de R$ 50,00 -> 100 pontos  (o broker entregou 2 vezes)
  publicado evt-10 PedidoCriado
  falhou    evt-10 PedidoCriado -> fidelidade (tentativa 1: confirmação perdida na rede)
  entregue  evt-10 PedidoCriado -> fidelidade (tentativa 2)


**Correção: receptor idempotente** (*Idempotent Receiver* [3]). O assinante guarda o identificador dos eventos que já tratou e ignora as repetições, confirmando-as sem refazer o efeito. Note que o broker continua entregando duas vezes; quem resolve é o assinante.

In [16]:
class ReceptorIdempotente:
    def __init__(self, handler):
        self.handler = handler
        self.processados = set()

    def __call__(self, evento):
        if evento.id in self.processados:
            return                       # já tratado: confirma sem repetir o efeito
        self.handler(evento)
        self.processados.add(evento.id)


pontos_idempotente, entregas, broker = cenario_duplicata(
    lambda f: ReceptorIdempotente(assinante_fidelidade(f)))
print(f"Pedido de R$ 50,00 -> {pontos_idempotente} pontos  (o broker entregou {entregas} vezes)")
broker.mostrar_log()

Pedido de R$ 50,00 -> 50 pontos  (o broker entregou 2 vezes)
  publicado evt-11 PedidoCriado
  falhou    evt-11 PedidoCriado -> fidelidade (tentativa 1: confirmação perdida na rede)
  entregue  evt-11 PedidoCriado -> fidelidade (tentativa 2)


### 3.2 Ordem trocada

A cliente compra e cancela logo em seguida. As duas mensagens entram na fila do painel na ordem certa: `PedidoCriado` e depois `PedidoCancelado`. Mas, na primeira tentativa de gravar `PedidoCriado`, o banco do painel não responde. A nova tentativa vai para o **fim** da fila, atrás do cancelamento.

In [17]:
def banco_falha_na_primeira(handler, tipo):
    "Simula o banco do painel sem responder na primeira gravação de um tipo de evento."
    ja_falhou = set()
    def tratar(evento):
        if evento.tipo == tipo and evento.id not in ja_falhou:
            ja_falhou.add(evento.id)
            raise ServicoIndisponivel("banco do painel não respondeu")
        handler(evento)
    return tratar


def cenario_ordem(montar_assinante):
    broker, painel = Broker(), Painel()
    loja = LojaComEventos(Estoque({"caneca": 10}), broker)
    tratar = banco_falha_na_primeira(montar_assinante(painel), "PedidoCriado")
    broker.assinar("PedidoCriado", "painel", tratar)
    broker.assinar("PedidoCancelado", "painel", tratar)

    pedido = novo_pedido()
    loja.finalizar_pedido(pedido)
    loja.cancelar_pedido(pedido)
    broker.entregar_pendentes()
    return painel.status[pedido["id"]], broker


tela_ingenua, broker = cenario_ordem(assinante_painel)
print(f"Situação real do pedido: cancelado")
print(f"O que a tela mostra:     {tela_ingenua}")
broker.mostrar_log()

Situação real do pedido: cancelado
O que a tela mostra:     confirmado
  publicado evt-12 PedidoCriado
  publicado evt-13 PedidoCancelado
  falhou    evt-12 PedidoCriado -> painel (tentativa 1: banco do painel não respondeu)
  entregue  evt-13 PedidoCancelado -> painel (tentativa 1)
  entregue  evt-12 PedidoCriado -> painel (tentativa 2)


O painel aplicou o cancelamento e depois a criação, e a tela mostra **"confirmado" para um pedido cancelado**.

**Correção: versão do evento.** A loja já incrementa `versao` a cada mudança do pedido e manda esse número no evento. O painel guarda a última versão que aplicou e ignora qualquer evento mais antigo.

In [18]:
def assinante_painel_versionado(painel):
    ultima_versao = {}
    def tratar(evento):
        d = evento.dados
        if d["versao"] <= ultima_versao.get(d["id"], 0):
            return                       # evento mais antigo que o último aplicado
        painel.registrar(d["id"], STATUS[evento.tipo])
        ultima_versao[d["id"]] = d["versao"]
    return tratar


tela_versionada, broker = cenario_ordem(assinante_painel_versionado)
print(f"Situação real do pedido: cancelado")
print(f"O que a tela mostra:     {tela_versionada}")
broker.mostrar_log()

Situação real do pedido: cancelado
O que a tela mostra:     cancelado
  publicado evt-14 PedidoCriado
  publicado evt-15 PedidoCancelado
  falhou    evt-14 PedidoCriado -> painel (tentativa 1: banco do painel não respondeu)
  entregue  evt-15 PedidoCancelado -> painel (tentativa 1)
  entregue  evt-14 PedidoCriado -> painel (tentativa 2)


O broker entregou exatamente na mesma ordem errada. A diferença é que o assinante não confia mais nessa ordem.

Nenhuma dessas duas falhas existia na versão acoplada, onde cada chamada acontece uma vez e na sequência do código. Elas são o preço da entrega assíncrona com novas tentativas.

## 4. Consequência mensurável

As verificações abaixo fazem o notebook falhar se algum resultado mudar. Assim, a tabela do texto e o código não podem discordar sem que alguém perceba.

In [19]:
# Seção 2: o que o estilo entrega
acoplada, eventos = resultados["Serviços que a loja precisa conhecer"]
assert (acoplada, eventos) == (4, 2)
assert t_eventos < t_acoplada / 10
assert tentativas_email == 3
assert pontos == 150

# Seção 3: o que acontece quando o assinante ignora a terceira restrição
assert (pontos_ingenuo, pontos_idempotente) == (100, 50)
assert (tela_ingenua, tela_versionada) == ("confirmado", "cancelado")

print(f"{'Cenário':28} | {'Assinante ingênuo':22} | {'Assinante protegido':22} | Correto")
print("-" * 90)
print(f"{'Entrega duplicada (R$ 50)':28} | {str(pontos_ingenuo) + ' pontos':22} | "
      f"{str(pontos_idempotente) + ' pontos':22} | 50 pontos")
print(f"{'Compra seguida de cancelamento':28} | {tela_ingenua:22} | {tela_versionada:22} | cancelado")
print("\nTodas as verificações passaram.")

Cenário                      | Assinante ingênuo      | Assinante protegido    | Correto
------------------------------------------------------------------------------------------
Entrega duplicada (R$ 50)    | 100 pontos             | 50 pontos              | 50 pontos
Compra seguida de cancelamento | confirmado             | cancelado              | cancelado

Todas as verificações passaram.


## 5. Conclusão

Com eventos, a loja deixou de conhecer quem reage à venda, passou a responder em uma fração do tempo e parou de cair junto com o e-mail ou a fidelidade. O preço apareceu na seção 3: com entrega "pelo menos uma vez", cada assinante precisa tolerar duplicatas e ordem trocada, e uma falha que antes era visível agora vai para a dead letter. É o mesmo par de forças que Garlan e Shaw apontavam na invocação implícita [6]: ganha-se evolução e perde-se controle sobre quem responde, quando e em que ordem.

## 6. Para pensar

Use o próprio notebook para testar as suas respostas.

1. O estoque ficou fora do broker. Transforme-o em assinante de `PedidoCriado` e simule dois clientes comprando a última caneca. O que cada um recebe como resposta? Quando o sistema descobre o problema?
2. Mude `max_tentativas` para 1 e repita o cenário do e-mail fora do ar (seção 2.5). Uma queda de poucos segundos no servidor passaria a ter que efeito? E com `max_tentativas` muito alto?
3. O `ReceptorIdempotente` guarda os identificadores em memória. O que acontece se o serviço de fidelidade reiniciar entre a primeira e a segunda entrega? Onde esses identificadores deveriam ficar?
4. A correção por versão funciona porque só a loja altera o pedido e sempre incrementa `versao`. E se dois serviços diferentes publicassem eventos sobre o mesmo pedido?
5. Um colega propõe publicar o evento `CalcularFrete` e esperar a resposta antes de confirmar a venda. Aplique o teste do nome do evento, visto no texto. Isso deveria ser evento?

---

### Referências

A numeração é a mesma da página do tópico, para que `[1]` signifique a mesma coisa nos dois lugares.

[1] P. Th. Eugster, P. A. Felber, R. Guerraoui e A.-M. Kermarrec, "The Many Faces of Publish/Subscribe", *ACM Computing Surveys*, vol. 35, n. 2, pp. 114–131, 2003. DOI 10.1145/857076.857078.

[2] M. Richards e N. Ford, *Fundamentals of Software Architecture: An Engineering Approach*. O'Reilly, 2020, cap. 14.

[3] G. Hohpe e B. Woolf, *Enterprise Integration Patterns: Designing, Building, and Deploying Messaging Solutions*. Addison-Wesley, 2003, caps. 4 e 10.

[4] M. Fowler, "What do you mean by 'Event-Driven'?", 2017. Disponível em: <https://martinfowler.com/articles/201701-event-driven.html>.

[6] D. Garlan e M. Shaw, "An Introduction to Software Architecture", em *Advances in Software Engineering and Knowledge Engineering*, pp. 1–39, 1993, seç. 3.3. DOI 10.1142/9789812798039_0001.

[11] M. Kleppmann, *Designing Data-Intensive Applications*. O'Reilly, 2017, cap. 11.

---

Conteúdo sob CC BY-NC 4.0, uso livre com crédito. Código sob MIT.